# Compute zonal statistics on the Net Primary Production dataset

In this notebook, we will compute zonal statistics on the Net Primary Production (NPP) dataset. The NPP dataset is a raster dataset that contains the Net Primary Production values for the entire globe. We will compute the mean values per year for each country.

## Setup

### Library import

In [ ]:
import geopandas as gpd
import pandas as pd
import regionmask
import xarray as xr
from dask.distributed import Client
from exactextract import exact_extract
from shapely.geometry import LineString, MultiPolygon, Polygon
from shapely.ops import split

**Start Dask Client for Dashboard**

In [ ]:
client = Client()
client  # noqa: B018

### Utils

In [ ]:
def add_mask_to_dataset(gdf_region, ds):
    """
    Add a mask to a dataset based on a region defined by a GeoDataFrame.

    Parameters
    ----------
    gdf_region : geopandas.GeoDataFrame
        The GeoDataFrame defining the region.
    ds : xarray.Dataset
        The dataset to be masked.
    """
    xmin_180, ymin, xmax_180, ymax = (
        gdf_region.to_crs("+proj=latlong +datum=WGS84 +lon_0=180")["geometry"].iloc[0].bounds
    )

    geom = gdf_region["geometry"].iloc[0]
    xmin, ymin, xmax, ymax = geom.bounds

    # Take care of the antimeridian
    if (
        not round(xmin_180) <= -179
        and not round(xmax_180) >= 179
        and round(xmin) <= -175
        and round(xmax) >= 175
    ):
        # Split the geometry with the antimeridian.
        gdf_split = split_geometry_with_antimeridian(gdf_region)

        ds_list = []
        for side in ["left", "right"]:
            gdf_side = gdf_split[gdf_split["side"] == side].drop(columns="side")
            geom = gdf_side["geometry"].iloc[0]
            # Rasterize vector data
            ds_masked = rasterize_region(geom, ds)

            ds_list.append(ds_masked)

        # Combine the two datasets using combine_nested
        ds_masked = xr.combine_nested([ds_list], concat_dim=["x", "y"])
    else:
        # Rasterize vector data
        ds_masked = rasterize_region(geom, ds)

    return ds_masked


def rasterize_region(geom, ds):
    """
    Rasterize a region defined by a geometry.

    Parameters
    ----------
    geom : shapely.geometry
        The geometry defining the region.
    ds : xarray.Dataset
        The dataset to be masked.
    """
    region = regionmask.Regions([geom])
    # Filter the dataset to the region of interest
    bbox = geom.bounds
    ds_region = ds.sel(x=slice(bbox[0], bbox[2]), y=slice(bbox[3], bbox[1]))
    # Create the mask
    mask = region.mask(ds_region.x, ds_region.y)
    # Add mask as a new variable into the xarray.Dataset
    ds_masked = ds_region
    ds_masked["region"] = mask

    return ds_masked


def split_geometry_with_antimeridian(gdf: gpd.GeoDataFrame):
    """
    Split a GeoDataFrame with a geometry crossing the antimeridian into two GeoDataFrames.
    The GeoDataFrame should contain only one geometry.
    The geometry should be a Polygon or a MultiPolygon.
    The function returns two GeoDataFrames, one for each side of the antimeridian.
    """
    # Define the cutting line
    line = LineString([(0, -90), (0, 90)])

    # Reproject the GeoDataFrame to WGS84 datum with the prime meridian at 180 degrees longitude.
    gdf_proj = gdf.to_crs("+proj=latlong +datum=WGS84 +lon_0=180")
    geometry = gdf_proj["geometry"].iloc[0]

    result = split(geometry, line)

    if type(geometry) == MultiPolygon:
        polygons_left_side = []
        polygons_right_side = []
        for geom in result.geoms:
            if geom.centroid.x <= line.coords[0][0]:
                polygons_left_side.append(geom)
            else:
                polygons_right_side.append(geom)

        geometry_left = MultiPolygon(polygons_left_side)
        geometry_right = MultiPolygon(polygons_right_side)

    elif type(geometry) == Polygon:
        for geom in result.geoms:
            if geom.centroid.x <= line.coords[0][0]:
                geometry_left = geom
            else:
                geometry_right = geom

    gdf_list = []
    for side, geometry in {"left": geometry_left, "right": geometry_right}.items():
        gdf_tmp = gdf_proj.copy()
        gdf_tmp["geometry"] = geometry
        gdf_tmp["side"] = side
        gdf_list.append(gdf_tmp)

    gdf_split = pd.concat(gdf_list)

    multipolygon = gdf_split["geometry"].iloc[1]
    updated_multipolygon = shift_lon_coor(multipolygon, delta=-180)

    gdf_split = gdf_split.to_crs("EPSG:4326")
    gdf_split.loc[1, "geometry"] = updated_multipolygon

    return gdf_split.iloc[:2]


def shift_lon_coor(multipolygon, delta=-180):
    """
    Shift the longitude coordinates of a MultiPolygon by a given delta.
    """
    updated_polygons = []
    for polygon in multipolygon.geoms:
        updated_coords = []
        coords = list(polygon.exterior.coords)
        updated_coords = [(lon + delta, lat) for lon, lat in coords]
        updated_polygon = Polygon(updated_coords)
        updated_polygons.append(updated_polygon)

    return MultiPolygon(updated_polygons)

## Load data
### Raster Data
#### MODIS/Terra Net Primary Production

In [ ]:
ds = (
    xr.open_dataset(
        "../data/processed/MOD17A3HGF/MOD17A3HGF_NPP_cog_nodata_reproj_2001.tif",
        chunks={"x": 8640, "y": 3600},
    )
    .squeeze()
    .drop_vars(["band", "spatial_ref"])
)
ds

### Vector Data
#### Country Borders – UN

In [ ]:
data_path = "../data/processed/countries.parquet"
countries = gpd.read_parquet(data_path)
countries.head()

## Zonal statistics
### With [`regionmask`](https://regionmask.readthedocs.io/en/stable/)
`regionmask` is used to rasterize each country and then compute the zonal statistics with xarray.

In [ ]:
gdf_region = countries[countries["ISO3_CODE"] == "ESP"]

ds_masked = add_mask_to_dataset(gdf_region, ds)

# Compute mean value
mean_value = ds_masked["band_data"].where(ds_masked["region"] == 0).mean(["x", "y"])
mean_value.compute()

### With [`exactextract`](https://github.com/isciences/exactextract?tab=readme-ov-file#exactextract)
`exactextract` provides a fast and accurate algorithm for summarizing values in the portion of a raster dataset that is covered by a polygon, often referred to as zonal statistics. Unlike other zonal statistics implementations, it takes into account raster cells that are partially covered by the polygon.

In [ ]:
gdf_region = countries[countries["ISO3_CODE"] == "ESP"]

exact_extract(rast=ds["band_data"], vec=gdf_region, ops=["mean", "min", "max"])